<a href="https://colab.research.google.com/github/mmilannaik/bostonhousepricing/blob/main/W16S3_SQL_Window_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Install pandasql
!pip install pandasql --quiet

# 2. Load libraries and your CSV into a pandas DataFrame
import pandas as pd
from pandasql import sqldf

# 3. Create a helper to run SQL against any DataFrame in your notebook
pysqldf = lambda query: sqldf(query, globals())

  Preparing metadata (setup.py) ... done


In [3]:
ipl = pd.read_csv('/content/W16S2_IPL_Ball_by_Ball_2008_2022.csv')
ipl.head(2)

,ID,innings,overs,ballnumber,batter,bowler,non-striker,extra_type,batsman_run,extras_run,total_run,non_boundary,isWicketDelivery,player_out,kind,fielders_involved,BattingTeam
0,1312200,1,0,1,YBK Jaiswal,Mohammed Shami,JC Buttler,NaN,0,0,0,0,0,NaN,NaN,NaN,Rajasthan Royals
1,1312200,1,0,2,YBK Jaiswal,Mohammed Shami,JC Buttler,legbyes,0,1,1,0,0,NaN,NaN,NaN,Rajasthan Royals


In [4]:
ipl.columns

Index(['ID', 'innings', 'overs', 'ballnumber', 'batter', 'bowler',
       'non-striker', 'extra_type', 'batsman_run', 'extras_run', 'total_run',
       'non_boundary', 'isWicketDelivery', 'player_out', 'kind',
       'fielders_involved', 'BattingTeam'],
      dtype='object')

# Q1 : Find IPL Batter based on team ans runs scored by them

In [5]:
pysqldf('''
SELECT * FROM(
SELECT BattingTeam,batter,SUM(batsman_run) AS 'total_runs',
DENSE_RANK() OVER(PARTITION BY BattingTeam ORDER BY SUM(batsman_run) DESC) AS 'rank'
FROM ipl
GROUP BY BattingTeam,batter) t
WHERE t.rank <6
ORDER BY t.BattingTeam,t.rank
''')

,BattingTeam,batter,total_runs,rank
0,Chennai Super Kings,SK Raina,4695,1
1,Chennai Super Kings,MS Dhoni,4404,2
2,Chennai Super Kings,F du Plessis,2721,3
3,Chennai Super Kings,AT Rayudu,1774,4
4,Chennai Super Kings,MEK Hussey,1768,5
...,...,...,...,...
85,Sunrisers Hyderabad,DA Warner,4016,1
86,Sunrisers Hyderabad,S Dhawan,2518,2
87,Sunrisers Hyderabad,KS Williamson,2105,3
88,Sunrisers Hyderabad,MK Pandey,1346,4


# Q2 : Find Runs scored by Virat Kohli in his 50th,100th ,200th Match (CUMSUM & AVg)

In [19]:
pysqldf('''

SELECT * FROM (
SELECT ROW_NUMBER() OVER(ORDER BY ID) AS 'match_no',match_run,career_run,career_avg
FROM
(
SELECT ID,SUM(batsman_run) AS 'match_run',
SUM(SUM(batsman_run)) OVER( ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS'career_run',
AVG(SUM(batsman_run)) OVER( ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS'career_avg'
FROM ipl
WHERE batter = 'V Kohli'
GROUP BY ID) t
)t2
WHERE match_no = 50 OR match_no = 100 or match_no = 200
ORDER BY match_no



''')

,match_no,match_run,career_run,career_avg
0,50,11,1131,22.62
1,100,13,2650,26.50
2,200,41,6334,31.67


# Q3 : Find virat kphli running average after every match

In [22]:
pysqldf('''

SELECT * FROM (
SELECT ROW_NUMBER() OVER(ORDER BY ID) AS 'match_no',match_run,career_run,rolling_avg
FROM
(
SELECT ID,SUM(batsman_run) AS 'match_run',
SUM(SUM(batsman_run)) OVER( ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS'career_run',
AVG(SUM(batsman_run)) OVER( ROWS BETWEEN 5 PRECEDING AND CURRENT ROW) AS'rolling_avg'
FROM ipl
WHERE batter = 'V Kohli'
GROUP BY ID) t
)t2
ORDER BY match_no



''')

,match_no,match_run,career_run,rolling_avg
0,1,1,1,1.000000
1,2,23,24,12.000000
2,3,13,37,12.333333
3,4,12,49,12.250000
4,5,1,50,10.000000
...,...,...,...,...
210,211,0,6509,16.166667
211,212,20,6529,19.500000
212,213,73,6602,31.666667
213,214,25,6627,34.333333


# Q4: Percent of Total

In [29]:
orders = pd.read_csv('/content/W15S2_orders.csv')
order_d = pd.read_csv('/content/W15S2_order_details.csv')
orders.head(2)

,order_id,user_id,r_id,amount,date,partner_id,delivery_time,delivery_rating,restaurant_rating
0,1001,1,1,550,2022-05-10,1,25,5,3.0
1,1002,1,2,415,2022-05-26,1,19,5,2.0


In [25]:
order_d.head(2)

,id,order_id,f_id
0,1,1001,1
1,2,1001,3


In [37]:
pysqldf(
    '''
    SELECT f_id,
    (total_value*1.0/SUM(total_value) OVER())*100 as 'percent_of_total'
    FROM
    (
    SELECT f_id, SUM(amount) AS 'total_value'
    FROM orders
    INNER JOIN order_d ON orders.order_id = order_d.order_id
    WHERE r_id = 1
    GROUP BY f_id
    ) t

    '''
)

,f_id,percent_of_total
0,1,46.212121
1,2,14.393939
2,3,39.393939


In [39]:
yt_view = pd.read_csv('/content/W16S3_youtube.csv')
yt_view.head(2)

,date,views
0,2019-01-01,2095
1,2019-01-02,9889


# Q5: Month by month views

In [45]:
pysqldf(
    '''
   SELECT
   strftime('%Y',date) as year,
   strftime('%m',date) as month,
   SUM(views) AS 'views',
   LAG(SUM(views)) OVER(ORDER BY strftime('%Y',date),strftime('%m',date)) As pre_views,
   (SUM(views)*1.0- LAG(SUM(views)) OVER(ORDER BY strftime('%Y',date),strftime('%m',date)))/LAG(SUM(views)) OVER(ORDER BY strftime('%Y',date),strftime('%m',date))*100 As 'perc_prev'
   FROM yt_view
   GROUP BY strftime('%Y',date),strftime('%m',date)
   ORDER BY year,month
'''
)

,year,month,views,pre_views,perc_prev
0,2019,01,157817,NaN,NaN
1,2019,02,143387,157817.0,-9.143502
2,2019,03,176075,143387.0,22.797046
3,2019,04,155699,176075.0,-11.572341
4,2019,05,155165,155699.0,-0.342969
...,...,...,...,...,...
324,2046,01,148295,153371.0,-3.309622
325,2046,02,125468,148295.0,-15.392967
326,2046,03,152955,125468.0,21.907578
327,2046,04,140896,152955.0,-7.884018


# Q6: Week by Week views

In [46]:
pysqldf(
    '''
   SELECT
    (views-LAG(views,7) OVER(ORDER BY strftime('%Y',date))/LAG(views,7) OVER(ORDER BY strftime('%Y',date)
   FROM yt_view
'''
)

PandaSQLException: (sqlite3.OperationalError) near "FROM": syntax error
[SQL: 
   SELECT 
    (views-LAG(views,7) OVER(ORDER BY strftime('%Y',date))/LAG(views,7) OVER(ORDER BY strftime('%Y',date)
   FROM yt_view 
]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [47]:
pysqldf(
    """
    SELECT
        date,
        views,
        ((views - LAG(views, 7) OVER (ORDER BY strftime('%Y', date), strftime('%m', date), strftime('%d', date))) * 1.0
            / LAG(views, 7) OVER (ORDER BY strftime('%Y', date), strftime('%m', date), strftime('%d', date))
        ) AS week_over_week_growth
    FROM yt_view
    """
)


,date,views,week_over_week_growth
0,2019-01-01,2095,NaN
1,2019-01-02,9889,NaN
2,2019-01-03,2346,NaN
3,2019-01-04,517,NaN
4,2019-01-05,81,NaN
...,...,...,...
9995,2046-05-14,7402,-0.111938
9996,2046-05-15,3531,10.888889
9997,2046-05-16,2674,-0.712071
9998,2046-05-17,8686,10.882353


# Q7 : Percentile & Quantile

In [48]:
studs = pd.read_csv('/content/W16S1_student_marks.csv')
studs.head(2)

,name,branch,marks
0,Nitish,EEE,82
1,Rishabh,EEE,91


In [49]:
studs.shape

(16, 3)

In [54]:
pysqldf(
  '''
SELECT AVG(marks) as 'median_marks'
FROM
(
  SELECT marks FROM studs
  ORDER BY marks
  LIMIT 2-(SELECT COUNT(*) FROM studs)%2
  OFFSET (SELECT (COUNT(*)-1)/2 FROM studs)
) t
'''
)

,median_marks
0,79.5


In [56]:
pysqldf(
    '''
  SELECT marks FROM studs
  ORDER BY marks
  LIMIT 2-(SELECT COUNT(*) FROM studs)%2
  OFFSET (SELECT (COUNT(*)-1)/2 FROM studs)
'''
)

,marks
0,78
1,81
